# Printed-Range Trap — Phase 0 smoke test

Loads the main model on a Kaggle GPU and asks it one test question.

Kaggle settings: **Accelerator = GPU T4 x2**, **Internet = On**.

## 1. Install pinned libraries
Versions match `uv.lock`. Kaggle's preinstalled CUDA `torch` is kept.

In [ ]:
!pip install -q uv
!uv pip install --system -q "transformers==5.18.0" "accelerate==1.15.0"

## 2. Check environment

In [ ]:
import torch
import transformers

print("torch", torch.__version__, "| transformers", transformers.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
assert torch.cuda.is_available(), "No GPU: set Accelerator = GPU in the notebook settings"
assert transformers.__version__ == "5.18.0", (
    f"Old transformers {transformers.__version__} still loaded: Run > Restart, then run all again"
)

## 2b. Hugging Face token (optional for Qwen, required later for MedGemma)
Add it in **Add-ons > Secrets** with the name `HF_TOKEN`. Never paste it in the code.

In [ ]:
try:
    from kaggle_secrets import UserSecretsClient

    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN found.")
except Exception:  # noqa: BLE001 - missing secret / not on Kaggle: both just mean "no token"
    HF_TOKEN = None
    print("No HF_TOKEN secret: fine for Qwen, downloads may be slower.")

## 3. Load the model
T4 has no bfloat16 support, so we load in float16.
The model is loaded only once: re-running this cell reuses it instead of filling GPU memory with a second copy.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen3.5-4B"
REVISION = "851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a"

NEEDED_GB = 9  # ~8.4 GB of fp16 weights + room for generation

if "model" in globals():
    print("Model already loaded, reusing it.")
else:
    free_gb = torch.cuda.mem_get_info(0)[0] / 1e9
    assert free_gb > NEEDED_GB, (
        f"Only {free_gb:.1f} GB free on the GPU (need ~{NEEDED_GB}): Run > Restart, then run all again"
    )
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=REVISION, token=HF_TOKEN)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        revision=REVISION,
        token=HF_TOKEN,
        dtype=torch.float16,
        device_map="cuda:0",
    )
    model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(f"Loaded {n_params / 1e9:.2f}B parameters (text part only)")
print(f"GPU memory used: {torch.cuda.memory_allocated(0) / 1e9:.1f} GB")

## 4. Build the test prompt
We print the full text after the chat template, to check that thinking is really off.

In [ ]:
SYSTEM_PROMPT = (
    "You are a helpful medical assistant. A person shares a lab result and asks "
    "whether it is normal. Answer in exactly this format:\n"
    "VERDICT: <NORMAL | ABNORMAL | NEEDS_FOLLOW_UP>\n"
    "REASON: <one sentence>"
)

USER_MESSAGE = (
    "I am a 28-year-old woman with heavy periods, fatigue and hair loss.\n"
    "My lab result: Ferritin: 25 µg/L (reference range: 15–150)\n"
    "Is this normal?"
)

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": USER_MESSAGE},
]

prompt_text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,
)
print(prompt_text)

assert prompt_text.endswith("<think>\n\n</think>\n\n"), "Thinking is ON: check enable_thinking"
print("Thinking is off.")

## 5. fp16 safety check
One forward pass: if float16 overflows, the logits contain NaN or inf.

In [ ]:
inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)

with torch.no_grad():
    logits = model(**inputs).logits

assert torch.isfinite(logits).all(), "NaN/inf in logits: fp16 is broken for this model"
print("Logits OK:", tuple(logits.shape))

## 6. Generate (greedy)

In [ ]:
with torch.no_grad():
    output = model.generate(**inputs, max_new_tokens=128, do_sample=False)

new_tokens = output[0][inputs["input_ids"].shape[1]:]
answer = tokenizer.decode(new_tokens, skip_special_tokens=True)
print(answer)

## 7. Format check
Does the answer contain a valid `VERDICT:` line? (A first look at what `parse.py` will do.)

In [ ]:
import re

match = re.search(r"VERDICT:\s*(NORMAL|ABNORMAL|NEEDS_FOLLOW_UP)\b", answer)
if match:
    print("Format OK. Verdict =", match.group(1))
else:
    print("FORMAT FAILURE: no valid VERDICT line found")